In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
from distgen import Generator

## N-dimensional Gaussian Distribution Example

Load a 6D Gaussian configuration from a YAML file and verify that the output particle distribution reproduces the input covariance matrix to machine precision.

In [2]:
D = Generator('data/gaussian_nd.in.yaml', verbose=1)

In [3]:
P = D.run()

Distribution format: None
Output file: None

Creating beam distribution....
   Beam starting from: time
   Total charge: 10 pC.
   Number of macroparticles: 200000.
   Assuming longitudinally polarized electrons...
   N-dimensional Gaussian distribution
      centroid, sigmas: 
         avg_x = 5E-06 m, sigma_x = 0.001 m
         avg_y = 8E-06 m, sigma_y = 0.002 m
         avg_z = 0.001 m, sigma_z = 0.003 m
         avg_px = 10000 eV/c, sigma_px = 1000 eV/c
         avg_py = 20000 eV/c, sigma_py = 2000 eV/c
         avg_pz = 1E+09 eV/c, sigma_pz = 5000 eV/c
      covariance matrix:
         1e-06, -1e-07, 0, 0, 0, 0
         -1e-07, 4e-06, 0, 0, 0, 0
         0, 0, 9e-06, 0, 0, 0
         0, 0, 0, 1e+06, 0, 0
         0, 0, 0, 0, 4e+06, 0
         0, 0, 0, 0, 0, 2.5e+07
   Time start: fixing all particle time values to start time: 0 s.
      Setting avg_t -> 0 s.
...done. Time Elapsed: 170.184 ms.

   Created particles in .particles: 
   ParticleGroup with 200000 particles with total c

### Compare output covariance to input

The input covariance matrix (in SI base units: meters for positions, eV/c for momenta) is compared against the sample covariance from the generated particles.

In [4]:
coords = ['x', 'y', 'z', 'px', 'py', 'pz']

# Input covariance matrix (from the YAML, in SI units: m for positions, eV/c for momenta)
cov_input = np.array(D._input['nd_gaussian_dist']['cov_matrix'], dtype=float)
print("Input covariance matrix:")
with np.printoptions(linewidth=200, formatter={'float': lambda x: f'{x:12.4e}'}):
    print(cov_input)

Input covariance matrix:
[[  1.0000e-06  -1.0000e-07   0.0000e+00   0.0000e+00   0.0000e+00   0.0000e+00]
 [ -1.0000e-07   4.0000e-06   0.0000e+00   0.0000e+00   0.0000e+00   0.0000e+00]
 [  0.0000e+00   0.0000e+00   9.0000e-06   0.0000e+00   0.0000e+00   0.0000e+00]
 [  0.0000e+00   0.0000e+00   0.0000e+00   1.0000e+06   0.0000e+00   0.0000e+00]
 [  0.0000e+00   0.0000e+00   0.0000e+00   0.0000e+00   4.0000e+06   0.0000e+00]
 [  0.0000e+00   0.0000e+00   0.0000e+00   0.0000e+00   0.0000e+00   2.5000e+07]]


In [5]:
# Output covariance matrix from generated particles
cov_output = P.cov(*coords)
print("Output covariance matrix:")
with np.printoptions(linewidth=200, formatter={'float': lambda x: f'{x:12.4e}'}):
    print(cov_output)

Output covariance matrix:
[[  1.0000e-06  -1.0000e-07   1.2931e-24   8.1475e-19  -7.0998e-19   1.1017e-13]
 [ -1.0000e-07   4.0000e-06  -2.4168e-24   1.9672e-18  -3.3921e-18  -2.4825e-13]
 [  2.1441e-24  -1.5658e-24   9.0000e-06  -1.3559e-18   1.1517e-17   6.8726e-13]
 [  3.4883e-19   1.7910e-18  -1.2474e-18   1.0000e+06   2.4195e-12  -1.3800e-07]
 [ -1.5531e-18  -4.3585e-18   1.4121e-17   3.3708e-12   4.0000e+06   1.2079e-07]
 [  1.1017e-13  -2.4824e-13   6.8727e-13  -1.3800e-07   1.2079e-07   2.5000e+07]]


### Precision metrics

We evaluate how well the output matches the input using:
1. **Relative error** on diagonal elements (variances)
2. **Correlation residuals** — normalize by σ_i σ_j to compare off-diagonal terms fairly across different units
3. **Symmetry check** — the output should be symmetric to machine precision

In [6]:
# 1. Relative error on diagonal (variances)
# Note: distgen's fix_avg_and_stds rescales using ddof=1 (N-1), while the whitening uses ddof=0 (N).
# This introduces a systematic factor of N/(N-1) - 1 = 1/(N-1) ≈ 5e-6 for N=200000.
diag_input = np.diag(cov_input)
diag_output = np.diag(cov_output)
rel_err_diag = (diag_output - diag_input) / diag_input

print("Relative error on variances (diagonal):")
for i, c in enumerate(coords):
    print(f"  σ²_{c}: input = {diag_input[i]:.3e}, output = {diag_output[i]:.3e}, rel err = {rel_err_diag[i]:.2e}")
print(f"\n  Max |relative error|: {np.max(np.abs(rel_err_diag)):.2e}")
print(f"  Expected from ddof mismatch (1/(N-1)): {1/(D._input['n_particle']-1):.2e}")

Relative error on variances (diagonal):
  σ²_x: input = 1.000e-06, output = 1.000e-06, rel err = 5.00e-06
  σ²_y: input = 4.000e-06, output = 4.000e-06, rel err = 5.00e-06
  σ²_z: input = 9.000e-06, output = 9.000e-06, rel err = 5.00e-06
  σ²_px: input = 1.000e+06, output = 1.000e+06, rel err = 5.00e-06
  σ²_py: input = 4.000e+06, output = 4.000e+06, rel err = 5.00e-06
  σ²_pz: input = 2.500e+07, output = 2.500e+07, rel err = 5.00e-06

  Max |relative error|: 5.00e-06
  Expected from ddof mismatch (1/(N-1)): 5.00e-06


In [7]:
# 2. Correlation residuals — separate "intentionally correlated" from "should be zero" terms
stds_in = np.sqrt(diag_input)  # sigmas from input cov
corr_output = cov_output / np.outer(stds_in, stds_in)  # output correlation matrix
corr_input = cov_input / np.outer(stds_in, stds_in)    # input correlation matrix

corr_residual = corr_output - corr_input
np.fill_diagonal(corr_residual, 0)

# Identify which off-diagonal entries should be zero vs non-zero
n = len(coords)
zero_mask = np.zeros((n, n), dtype=bool)
nonzero_mask = np.zeros((n, n), dtype=bool)
for i in range(n):
    for j in range(i+1, n):
        if cov_input[i, j] == 0:
            zero_mask[i, j] = zero_mask[j, i] = True
        else:
            nonzero_mask[i, j] = nonzero_mask[j, i] = True

max_zero_corr = np.max(np.abs(corr_residual[zero_mask])) if zero_mask.any() else 0
max_nonzero_corr = np.max(np.abs(corr_residual[nonzero_mask])) if nonzero_mask.any() else 0

print("Correlation residual matrix (corr_output - corr_input):")
print(np.array2string(corr_residual, precision=2))
print(f"\n  Max |residual| on 'should be zero' terms:     {max_zero_corr:.2e}")
print(f"  Max |residual| on intentionally-correlated terms: {max_nonzero_corr:.2e}")
print(f"\n  The 'should be zero' terms are at machine precision (~1e-14).")
print(f"  The 2.5e-7 residual on (x,y) is just 1/(N-1) × corr(x,y) from P.cov()'s ddof=1.")

Correlation residual matrix (corr_output - corr_input):
[[ 0.00e+00 -2.50e-07  4.31e-19  8.15e-19 -3.55e-19  2.20e-14]
 [-2.50e-07  0.00e+00 -4.03e-19  9.84e-19 -8.48e-19 -2.48e-14]
 [ 7.15e-19 -2.61e-19  0.00e+00 -4.52e-19  1.92e-18  4.58e-14]
 [ 3.49e-19  8.95e-19 -4.16e-19  0.00e+00  1.21e-18 -2.76e-14]
 [-7.77e-19 -1.09e-18  2.35e-18  1.69e-18  0.00e+00  1.21e-14]
 [ 2.20e-14 -2.48e-14  4.58e-14 -2.76e-14  1.21e-14  0.00e+00]]

  Max |residual| on 'should be zero' terms:     4.58e-14
  Max |residual| on intentionally-correlated terms: 2.50e-07

  The 'should be zero' terms are at machine precision (~1e-14).
  The 2.5e-7 residual on (x,y) is just 1/(N-1) × corr(x,y) from P.cov()'s ddof=1.


In [8]:
# 3. Why do raw covariance entries like cov(px,pz) = -1.38e-7 look non-zero?
#
# The error comes from floating-point quantization when adding large centroids.
# pz values are stored as (1e9 + fluctuation), where 1e9 is the centroid.
# The ULP (unit in last place) near 1e9 is: 1e9 * eps ≈ 2.2e-7.
# This quantization noise creates spurious covariance that scales as:
#    |cov_spurious(i, pz)| ~ sigma_i * ULP_pz / sqrt(N)

eps = np.finfo(float).eps
centroids = np.array([5e-6, 8e-6, 1e-3, 1e4, 2e4, 1e9])  # in SI units

print("Floating-point noise analysis for 'should be zero' entries involving pz:")
print(f"  pz centroid = 1 GeV/c = 1e9 eV/c")
print(f"  ULP at 1e9: eps × 1e9 = {eps * 1e9:.2e} eV/c")
print(f"  N = {D._input['n_particle']}")
print()

N = D._input['n_particle']
ULP_pz = eps * centroids[5]

print(f"  {'pair':<10} {'observed':>12} {'predicted':>12}   (predicted = σ_i × ULP_pz / √N)")
print(f"  {'-'*50}")
for i, c in enumerate(coords[:5]):
    observed = abs(cov_output[i, 5])
    predicted = stds_in[i] * ULP_pz / np.sqrt(N)
    print(f"  ({c}, pz)   {observed:12.2e} {predicted:12.2e}")

print(f"\n  These are all < {max_zero_corr:.0e} in normalized correlation — machine precision.")

Floating-point noise analysis for 'should be zero' entries involving pz:
  pz centroid = 1 GeV/c = 1e9 eV/c
  ULP at 1e9: eps × 1e9 = 2.22e-07 eV/c
  N = 200000

  pair           observed    predicted   (predicted = σ_i × ULP_pz / √N)
  --------------------------------------------------
  (x, pz)       1.10e-13     4.97e-13
  (y, pz)       2.48e-13     9.93e-13
  (z, pz)       6.87e-13     1.49e-12
  (px, pz)       1.38e-07     4.97e-07
  (py, pz)       1.21e-07     9.93e-07

  These are all < 5e-14 in normalized correlation — machine precision.


## Failure Mode Checking
---

Use of the Nd Gaussian function will limit what other distributions the user can specify.  Variables used in the Nd dist can't be specified by other distributions:

In [9]:

YAML="""n_particle: 200000
species: electron
nd_gaussian_dist:
    centroid:
        x: 0.005 mm
        y: 0.008 mm
        z: 1.0 mm
        px: 10 keV/c
        py: 20 keV/c
        pz: 1 GeV/c
    sigmas:
        x: 1 mm
        y: 2 mm
        z: 3 mm
        px: 1 keV/c
        py: 2 keV/c
        pz: 3 MeV/c
x_dist:
    type: gaussian
    avg_x: 2 mm
    sigma_x: 1 mm
random:
  type: hammersley
start:
  tstart: 0 sec
  type: time
total_charge: 10 pC
spin_polarization: 0.35
"""

try: 
    D = Generator(YAML)
    D.run()
except Exception as e:
    print("Error:", e)

Error: Variable x specified in both nd_gaussian_dist and as separate distribution.


In addition, the user cannot specifcy a cathode start and the momentum variables px, py, and/or pz in the Nd Gaussian:

In [10]:
YAML="""n_particle: 200000
species: electron
nd_gaussian_dist:
    centroid:
        x: 0.005 mm
        y: 0.008 mm
        z: 1.0 mm
        px: 10 keV/c
        py: 20 keV/c
        pz: 1 GeV/c
    sigmas:
        x: 1 mm
        y: 2 mm
        z: 3 mm
        px: 1 keV/c
        py: 2 keV/c
        pz: 3 MeV/c
random:
  type: hammersley
start:
  MTE: 120 meV
  type: cathode
total_charge: 10 pC
spin_polarization: 0.35
"""

try: 
    D = Generator(YAML)
    D.run()
except Exception as e:
    print("Error:", e)

Error: Momentum variable px cannot be specified in nd_gaussian_dist for cathode start.
